# Week 11: Simple Harmonic Motion (SHM) — PHASE 5: Oscillations & Waves

*📚 Physics I (PHY101) · ⏱️ 3 Hours · 👨‍🏫 Dr. Arif Solmaz*

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Define** simple harmonic motion and identify systems that exhibit SHM
2. **Derive** the equations of motion for spring-mass and pendulum systems
3. **Relate** displacement, velocity, and acceleration in SHM using phase relationships
4. **Calculate** the energy exchange between kinetic and potential energy during oscillation
5. **Explain** damped oscillations and their dependence on the damping coefficient
6. **Solve** SHM differential equations numerically using the Euler method and `scipy.integrate.odeint`
7. **Visualize** phase-space trajectories and energy diagrams for oscillating systems

## 🎯 Core Mastery Connection

SHM is what happens when you apply $F = ma$ to a restoring force $F = -kx$. The result is sinusoidal motion that you can fully predict from initial conditions. Diagram the forces on the oscillating mass, identify the restoring-force principle, write the equation of motion, and predict amplitude, frequency, and energy exchange. This is the core mastery workflow applied to systems that repeat.

---
## 1. Setup

Run this cell first to import all necessary libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.patches import FancyBboxPatch, Circle
from IPython.display import HTML, display, clear_output
import ipywidgets as widgets
from scipy.integrate import odeint

plt.rcParams.update({
    'figure.figsize': (10, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print("All libraries loaded successfully!")

---
## 2. What is Simple Harmonic Motion?

**Simple Harmonic Motion (SHM)** is a type of periodic motion where the restoring force is **directly proportional** to the displacement and acts in the **opposite direction**.

$$F = -kx$$

This leads to the differential equation:

$$m\ddot{x} + kx = 0 \quad \Longrightarrow \quad \ddot{x} + \omega^2 x = 0$$

where $\omega = \sqrt{k/m}$ is the **angular frequency**.

### The General Solution

$$x(t) = A\cos(\omega t + \phi)$$

| Quantity | Symbol | Expression | Unit |
|----------|--------|------------|------|
| Amplitude | $A$ | max displacement | m |
| Angular frequency | $\omega$ | $\sqrt{k/m}$ | rad/s |
| Period | $T$ | $2\pi/\omega$ | s |
| Frequency | $f$ | $1/T = \omega/(2\pi)$ | Hz |
| Phase constant | $\phi$ | from initial conditions | rad |

### Analogy: SHM as Circular Motion

Think of SHM as the **shadow** of uniform circular motion. If a ball moves in a circle of radius $A$ with angular speed $\omega$, its projection on the x-axis traces out SHM. This is why we use sine and cosine functions!

### Velocity and Acceleration

$$v(t) = -A\omega\sin(\omega t + \phi)$$

$$a(t) = -A\omega^2\cos(\omega t + \phi) = -\omega^2 x(t)$$

Key observation: Acceleration is always **proportional and opposite** to displacement.

---
## 3. Interactive Demo 1: Animated Spring-Mass Oscillation with Phase Space

This animation shows a mass on a spring oscillating back and forth. The right panel displays the **phase space** plot ($x$ vs $v$), which forms an ellipse for SHM.

In [ ]:
def spring_mass_animation(A=1.0, omega=2*np.pi, phi=0.0):
    """Animated spring-mass system with phase-space plot."""
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 5),
                                         gridspec_kw={'width_ratios': [2, 2, 1.5]})
    plt.close(fig)

    T = 2 * np.pi / omega
    t_total = 3 * T
    dt = 0.03
    t_arr = np.arange(0, t_total, dt)
    x_arr = A * np.cos(omega * t_arr + phi)
    v_arr = -A * omega * np.sin(omega * t_arr + phi)

    # --- Left panel: spring-mass ---
    ax1.set_xlim(-2.5, 2.5)
    ax1.set_ylim(-1, 1)
    ax1.set_aspect('equal')
    ax1.set_title('Spring-Mass System', fontsize=13, fontweight='bold')
    ax1.axvline(x=0, color='gray', ls='--', alpha=0.5)
    ax1.set_yticks([])
    ax1.set_xlabel('Position x (m)')

    wall = plt.Rectangle((-2.5, -0.5), 0.1, 1.0, color='gray')
    ax1.add_patch(wall)

    spring_line, = ax1.plot([], [], 'b-', lw=2)
    mass_patch = plt.Rectangle((0, -0.2), 0.4, 0.4, fc='royalblue', ec='navy', lw=2)
    ax1.add_patch(mass_patch)
    force_arrow = ax1.annotate('', xy=(0, 0), xytext=(0, 0),
                               arrowprops=dict(arrowstyle='->', color='red', lw=2))
    time_text = ax1.text(0, 0.7, '', fontsize=11, ha='center',
                         bbox=dict(boxstyle='round', fc='lightyellow'))

    def make_spring(x_start, x_end, n_coils=10, width=0.15):
        L = x_end - x_start
        xs = np.linspace(0, 1, n_coils * 20 + 1)
        spring_x = x_start + xs * L
        spring_y = width * np.sin(2 * np.pi * n_coils * xs)
        spring_y[0] = 0; spring_y[-1] = 0
        return spring_x, spring_y

    # --- Middle panel: x(t) and v(t) ---
    ax2.set_xlim(0, t_total)
    ax2.set_ylim(-A * omega * 1.3, A * omega * 1.3)
    ax2.set_title('x(t) and v(t)', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('x (m) / v (m/s)')
    x_line, = ax2.plot([], [], 'b-', lw=2, label='x(t)')
    v_line, = ax2.plot([], [], 'r-', lw=2, alpha=0.7, label='v(t)')
    dot_x, = ax2.plot([], [], 'bo', ms=8, zorder=5)
    dot_v, = ax2.plot([], [], 'ro', ms=8, zorder=5)
    ax2.legend(loc='upper right')

    # --- Right panel: phase space ---
    ax3.set_xlim(-A * 1.3, A * 1.3)
    ax3.set_ylim(-A * omega * 1.3, A * omega * 1.3)
    ax3.set_title('Phase Space', fontsize=13, fontweight='bold')
    ax3.set_xlabel('x (m)')
    ax3.set_ylabel('v (m/s)')
    ax3.set_aspect('equal')
    theta_full = np.linspace(0, 2 * np.pi, 200)
    ax3.plot(A * np.cos(theta_full), -A * omega * np.sin(theta_full),
             'gray', ls='--', alpha=0.4)
    phase_trail, = ax3.plot([], [], 'g-', lw=1.5, alpha=0.6)
    phase_dot, = ax3.plot([], [], 'go', ms=10, zorder=5)

    fig.tight_layout()

    def animate(i):
        t = t_arr[i]
        x = x_arr[i]
        v = v_arr[i]

        # Spring
        sx, sy = make_spring(-2.4, x - 0.2)
        spring_line.set_data(sx, sy)
        mass_patch.set_xy((x - 0.2, -0.2))

        # Force arrow
        force_arrow.xy = (x - 0.3 * np.sign(x), 0.0)
        force_arrow.xyann = (x, 0.0)
        force_arrow.set_visible(abs(x) > 0.05)

        time_text.set_text(f't = {t:.2f} s\nx = {x:.2f} m')

        # Time plots
        x_line.set_data(t_arr[:i+1], x_arr[:i+1])
        v_line.set_data(t_arr[:i+1], v_arr[:i+1])
        dot_x.set_data([t], [x])
        dot_v.set_data([t], [v])

        # Phase space
        trail_start = max(0, i - 80)
        phase_trail.set_data(x_arr[trail_start:i+1], v_arr[trail_start:i+1])
        phase_dot.set_data([x], [v])

        return spring_line, mass_patch, time_text, x_line, v_line, dot_x, dot_v, phase_trail, phase_dot

    ani = animation.FuncAnimation(fig, animate, frames=len(t_arr),
                                  interval=30, blit=False)
    return HTML(ani.to_jshtml())

display(spring_mass_animation(A=1.5, omega=2*np.pi, phi=0))

### Explore with Sliders

Use the sliders below to change the amplitude, angular frequency, and initial phase.

In [ ]:
def interactive_shm(A, omega, phi_deg):
    phi = np.radians(phi_deg)
    t = np.linspace(0, 4, 500)
    x = A * np.cos(omega * t + phi)
    v = -A * omega * np.sin(omega * t + phi)
    a = -A * omega**2 * np.cos(omega * t + phi)

    fig, axes = plt.subplots(2, 2, figsize=(12, 8))

    # x(t)
    axes[0, 0].plot(t, x, 'b-', lw=2)
    axes[0, 0].set_ylabel('x (m)', fontsize=12)
    axes[0, 0].set_title('Displacement', fontweight='bold')
    axes[0, 0].set_ylim(-3, 3)

    # v(t)
    axes[0, 1].plot(t, v, 'r-', lw=2)
    axes[0, 1].set_ylabel('v (m/s)', fontsize=12)
    axes[0, 1].set_title('Velocity', fontweight='bold')
    axes[0, 1].set_ylim(-20, 20)

    # a(t)
    axes[1, 0].plot(t, a, 'g-', lw=2)
    axes[1, 0].set_xlabel('Time (s)', fontsize=12)
    axes[1, 0].set_ylabel('a (m/s²)', fontsize=12)
    axes[1, 0].set_title('Acceleration', fontweight='bold')
    axes[1, 0].set_ylim(-120, 120)

    # Phase space
    axes[1, 1].plot(x, v, 'purple', lw=2)
    axes[1, 1].set_xlabel('x (m)', fontsize=12)
    axes[1, 1].set_ylabel('v (m/s)', fontsize=12)
    axes[1, 1].set_title('Phase Space', fontweight='bold')
    axes[1, 1].set_aspect('equal')

    T = 2 * np.pi / omega
    fig.suptitle(f'SHM: A={A:.1f} m, $\\omega$={omega:.1f} rad/s, T={T:.2f} s, $\\phi$={phi_deg:.0f}°',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(interactive_shm,
    A=widgets.FloatSlider(value=1.0, min=0.1, max=3.0, step=0.1, description='Amplitude A (m):',
                          style={'description_width': 'initial'}),
    omega=widgets.FloatSlider(value=2*np.pi, min=1.0, max=6*np.pi, step=0.5,
                              description='$\\omega$ (rad/s):', style={'description_width': 'initial'}),
    phi_deg=widgets.FloatSlider(value=0, min=0, max=360, step=15,
                                description='Phase $\\phi$ (deg):', style={'description_width': 'initial'})
);

---
## 4. The Simple Pendulum

A pendulum of length $L$ with a mass $m$ swings under gravity. For small angles ($\theta \ll 1$ rad):

$$\ddot{\theta} + \frac{g}{L}\theta = 0$$

This is SHM with $\omega = \sqrt{g/L}$ and period:

$$T = 2\pi\sqrt{\frac{L}{g}}$$

| Key Property | Spring-Mass | Simple Pendulum |
|-------------|-------------|------------------|
| Restoring force | $-kx$ | $-mg\sin\theta \approx -mg\theta$ |
| Angular frequency | $\sqrt{k/m}$ | $\sqrt{g/L}$ |
| Period depends on | mass $m$, spring constant $k$ | length $L$, gravity $g$ |
| Period independent of | amplitude (for SHM) | mass, amplitude (small angle) |

**Note:** The small-angle approximation $\sin\theta \approx \theta$ is accurate to within 1% for $\theta < 14°$.

---
## 5. Interactive Demo 2: Simple Pendulum Animation

Watch how the pendulum swings. Adjust the length and initial angle to see how the motion changes.

In [ ]:
def pendulum_animation(L=1.5, theta0_deg=30):
    """Animated simple pendulum with angle plot."""
    g = 9.81
    theta0 = np.radians(theta0_deg)

    # Solve full nonlinear ODE for accuracy
    def pendulum_ode(state, t):
        theta, omega = state
        return [omega, -g / L * np.sin(theta)]

    T_approx = 2 * np.pi * np.sqrt(L / g)
    t_span = np.linspace(0, 4 * T_approx, 600)
    sol = odeint(pendulum_ode, [theta0, 0], t_span)
    theta_arr = sol[:, 0]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 6),
                                    gridspec_kw={'width_ratios': [1, 1.5]})
    plt.close(fig)

    # --- Left: pendulum visual ---
    ax1.set_xlim(-L * 1.5, L * 1.5)
    ax1.set_ylim(-L * 1.5, 0.5)
    ax1.set_aspect('equal')
    ax1.set_title('Simple Pendulum', fontsize=14, fontweight='bold')
    ax1.plot(0, 0, 'ks', ms=10)  # pivot

    rod_line, = ax1.plot([], [], 'k-', lw=2)
    bob, = ax1.plot([], [], 'o', color='crimson', ms=20, mec='darkred', mew=2)
    trail, = ax1.plot([], [], 'r-', lw=1, alpha=0.3)
    info_text = ax1.text(-L * 1.3, 0.3, '', fontsize=11,
                          bbox=dict(boxstyle='round', fc='lightyellow'))

    trail_x, trail_y = [], []

    # --- Right: theta(t) ---
    ax2.set_xlim(0, t_span[-1])
    ax2.set_ylim(-np.degrees(theta0) * 1.5, np.degrees(theta0) * 1.5)
    ax2.set_xlabel('Time (s)', fontsize=12)
    ax2.set_ylabel('$\\theta$ (degrees)', fontsize=12)
    ax2.set_title('Angle vs Time', fontsize=14, fontweight='bold')
    theta_line, = ax2.plot([], [], 'crimson', lw=2)
    theta_dot, = ax2.plot([], [], 'o', color='crimson', ms=8)
    # Small angle comparison
    theta_linear = theta0 * np.cos(np.sqrt(g / L) * t_span)
    ax2.plot(t_span, np.degrees(theta_linear), 'b--', lw=1.5, alpha=0.5,
             label='Small-angle approx')
    ax2.legend(loc='upper right')

    fig.tight_layout()

    def animate(i):
        th = theta_arr[i]
        bx = L * np.sin(th)
        by = -L * np.cos(th)

        rod_line.set_data([0, bx], [0, by])
        bob.set_data([bx], [by])

        trail_x.append(bx)
        trail_y.append(by)
        if len(trail_x) > 100:
            trail_x.pop(0); trail_y.pop(0)
        trail.set_data(trail_x, trail_y)

        info_text.set_text(f'L = {L:.2f} m\nT = {T_approx:.2f} s\n$\\theta$ = {np.degrees(th):.1f}°')

        theta_line.set_data(t_span[:i+1], np.degrees(theta_arr[:i+1]))
        theta_dot.set_data([t_span[i]], [np.degrees(th)])

        return rod_line, bob, trail, info_text, theta_line, theta_dot

    ani = animation.FuncAnimation(fig, animate, frames=len(t_span),
                                  interval=30, blit=False)
    return HTML(ani.to_jshtml())

display(pendulum_animation(L=1.5, theta0_deg=30))

In [ ]:
def interactive_pendulum(L, theta0_deg):
    g = 9.81
    theta0 = np.radians(theta0_deg)
    T = 2 * np.pi * np.sqrt(L / g)

    def pend_ode(state, t):
        return [state[1], -g / L * np.sin(state[0])]

    t = np.linspace(0, 4 * T, 800)
    sol = odeint(pend_ode, [theta0, 0], t)
    theta_exact = sol[:, 0]
    theta_approx = theta0 * np.cos(np.sqrt(g / L) * t)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.plot(t, np.degrees(theta_exact), 'r-', lw=2, label='Full solution')
    ax1.plot(t, np.degrees(theta_approx), 'b--', lw=2, label='Small-angle')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Angle (degrees)')
    ax1.set_title(f'Pendulum: L={L:.1f} m, T={T:.2f} s', fontweight='bold')
    ax1.legend()

    error = np.abs(np.degrees(theta_exact - theta_approx))
    ax2.plot(t, error, 'g-', lw=2)
    ax2.set_xlabel('Time (s)')
    ax2.set_ylabel('Error (degrees)')
    ax2.set_title('Small-Angle Approximation Error', fontweight='bold')

    plt.tight_layout()
    plt.show()

widgets.interact(interactive_pendulum,
    L=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1,
                          description='Length L (m):', style={'description_width': 'initial'}),
    theta0_deg=widgets.FloatSlider(value=15, min=5, max=90, step=5,
                                   description='Initial angle (deg):', style={'description_width': 'initial'})
);

---
## 6. Energy in SHM

The total mechanical energy in SHM is conserved (no friction):

$$E_{\text{total}} = \frac{1}{2}kA^2 = \text{constant}$$

At any instant:

$$\text{KE} = \frac{1}{2}mv^2 = \frac{1}{2}kA^2\sin^2(\omega t + \phi)$$

$$\text{PE} = \frac{1}{2}kx^2 = \frac{1}{2}kA^2\cos^2(\omega t + \phi)$$

$$\text{KE} + \text{PE} = \frac{1}{2}kA^2$$

Think of it like water sloshing between two connected tanks: when one is full, the other is empty, but the total water never changes.

---
## 7. Interactive Demo 3: Energy Exchange Visualization

Watch KE and PE exchange in real time with both a line plot and animated bar chart.

In [ ]:
def energy_animation(A=1.0, k=10.0, m=1.0):
    """Animated energy exchange: KE vs PE bar chart and line plot."""
    omega = np.sqrt(k / m)
    T = 2 * np.pi / omega
    t_arr = np.linspace(0, 3 * T, 400)
    dt = t_arr[1] - t_arr[0]

    x_arr = A * np.cos(omega * t_arr)
    v_arr = -A * omega * np.sin(omega * t_arr)
    KE_arr = 0.5 * m * v_arr**2
    PE_arr = 0.5 * k * x_arr**2
    E_total = 0.5 * k * A**2

    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5),
                                         gridspec_kw={'width_ratios': [1, 2, 1.2]})
    plt.close(fig)

    # --- Bar chart ---
    bars = ax1.bar(['KE', 'PE'], [0, 0], color=['dodgerblue', 'orangered'],
                   edgecolor='black', lw=2)
    ax1.set_ylim(0, E_total * 1.2)
    ax1.set_ylabel('Energy (J)', fontsize=12)
    ax1.set_title('Energy Bars', fontweight='bold')
    ax1.axhline(y=E_total, color='green', ls='--', lw=2, label=f'E_total = {E_total:.2f} J')
    ax1.legend(fontsize=9)

    # --- Energy vs time ---
    ax2.set_xlim(0, t_arr[-1])
    ax2.set_ylim(0, E_total * 1.2)
    ax2.set_xlabel('Time (s)', fontsize=12)
    ax2.set_ylabel('Energy (J)', fontsize=12)
    ax2.set_title('Energy vs Time', fontweight='bold')
    ke_line, = ax2.plot([], [], 'dodgerblue', lw=2, label='KE')
    pe_line, = ax2.plot([], [], 'orangered', lw=2, label='PE')
    ax2.axhline(y=E_total, color='green', ls='--', lw=2, label='Total E')
    ax2.legend(loc='upper right')

    # --- Position indicator ---
    ax3.set_xlim(-A * 1.5, A * 1.5)
    ax3.set_ylim(-0.5, 0.5)
    ax3.set_aspect('equal')
    ax3.set_title('Mass Position', fontweight='bold')
    ax3.set_xlabel('x (m)')
    ax3.set_yticks([])
    ax3.axvline(0, color='gray', ls='--', alpha=0.5)
    pos_dot, = ax3.plot([], [], 'o', color='navy', ms=20)
    pos_text = ax3.text(0, 0.35, '', ha='center', fontsize=10)

    fig.tight_layout()

    def animate(i):
        ke_val = KE_arr[i]
        pe_val = PE_arr[i]

        bars[0].set_height(ke_val)
        bars[1].set_height(pe_val)

        ke_line.set_data(t_arr[:i+1], KE_arr[:i+1])
        pe_line.set_data(t_arr[:i+1], PE_arr[:i+1])

        pos_dot.set_data([x_arr[i]], [0])
        pos_text.set_text(f'x = {x_arr[i]:.2f} m')

        return bars[0], bars[1], ke_line, pe_line, pos_dot, pos_text

    ani = animation.FuncAnimation(fig, animate, frames=len(t_arr),
                                  interval=25, blit=False)
    return HTML(ani.to_jshtml())

display(energy_animation(A=1.0, k=10.0, m=1.0))

In [ ]:
def interactive_energy(A, k, m):
    omega = np.sqrt(k / m)
    T = 2 * np.pi / omega
    t = np.linspace(0, 3 * T, 500)
    x = A * np.cos(omega * t)
    v = -A * omega * np.sin(omega * t)
    KE = 0.5 * m * v**2
    PE = 0.5 * k * x**2
    E_total = 0.5 * k * A**2

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    ax1.fill_between(t, 0, KE, alpha=0.4, color='dodgerblue', label='KE')
    ax1.fill_between(t, KE, KE + PE, alpha=0.4, color='orangered', label='PE')
    ax1.axhline(y=E_total, color='green', ls='--', lw=2, label=f'Total = {E_total:.2f} J')
    ax1.set_xlabel('Time (s)')
    ax1.set_ylabel('Energy (J)')
    ax1.set_title('Stacked Energy Plot', fontweight='bold')
    ax1.legend()

    ax2.plot(x, KE, 'dodgerblue', lw=2, label='KE')
    ax2.plot(x, PE, 'orangered', lw=2, label='PE')
    ax2.axhline(y=E_total, color='green', ls='--', lw=2, label='Total')
    ax2.set_xlabel('x (m)')
    ax2.set_ylabel('Energy (J)')
    ax2.set_title('Energy vs Position', fontweight='bold')
    ax2.legend()

    fig.suptitle(f'k={k:.1f} N/m, m={m:.1f} kg, A={A:.1f} m, $\\omega$={omega:.2f} rad/s',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(interactive_energy,
    A=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1,
                          description='Amplitude (m):', style={'description_width': 'initial'}),
    k=widgets.FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0,
                          description='k (N/m):', style={'description_width': 'initial'}),
    m=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                          description='m (kg):', style={'description_width': 'initial'})
);

---
## 8. Damped Oscillations

In the real world, friction and air resistance cause oscillations to die out. The equation of motion for a **damped harmonic oscillator** is:

$$m\ddot{x} + b\dot{x} + kx = 0$$

where $b$ is the **damping coefficient**. Defining $\gamma = b/(2m)$ and $\omega_0 = \sqrt{k/m}$:

$$\ddot{x} + 2\gamma\dot{x} + \omega_0^2 x = 0$$

Three regimes:

| Regime | Condition | Behavior |
|--------|-----------|----------|
| **Underdamped** | $\gamma < \omega_0$ | Oscillates with decaying amplitude |
| **Critically damped** | $\gamma = \omega_0$ | Fastest return without oscillation |
| **Overdamped** | $\gamma > \omega_0$ | Slow exponential return |

For underdamped motion:

$$x(t) = A e^{-\gamma t}\cos(\omega_d t + \phi)$$

where $\omega_d = \sqrt{\omega_0^2 - \gamma^2}$ is the **damped angular frequency**.

---
## 9. Interactive Demo 4: Damped Oscillation

Adjust the damping coefficient and observe how the amplitude decays. The envelope $\pm A e^{-\gamma t}$ is shown in dashed lines.

In [ ]:
def interactive_damping(b, k, m, A):
    omega0 = np.sqrt(k / m)
    gamma = b / (2 * m)

    def damped_ode(state, t):
        x, v = state
        return [v, -2 * gamma * v - omega0**2 * x]

    t = np.linspace(0, 15, 2000)
    sol = odeint(damped_ode, [A, 0], t)
    x = sol[:, 0]
    v = sol[:, 1]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))

    # x(t)
    axes[0].plot(t, x, 'b-', lw=2, label='x(t)')
    envelope = A * np.exp(-gamma * t)
    axes[0].plot(t, envelope, 'r--', lw=1.5, alpha=0.7, label='Envelope')
    axes[0].plot(t, -envelope, 'r--', lw=1.5, alpha=0.7)
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('x (m)')
    axes[0].legend()

    # Regime label
    if gamma < omega0 * 0.99:
        regime = 'UNDERDAMPED'
        omega_d = np.sqrt(omega0**2 - gamma**2)
        regime_info = f'$\\omega_d$ = {omega_d:.2f} rad/s'
    elif gamma > omega0 * 1.01:
        regime = 'OVERDAMPED'
        regime_info = 'No oscillation'
    else:
        regime = 'CRITICALLY DAMPED'
        regime_info = 'Fastest return'

    axes[0].set_title(f'{regime}\n{regime_info}', fontweight='bold', fontsize=12)

    # Phase space
    axes[1].plot(x, v, 'purple', lw=1.5)
    axes[1].plot(x[0], v[0], 'go', ms=10, label='Start')
    axes[1].plot(x[-1], v[-1], 'ro', ms=10, label='End')
    axes[1].set_xlabel('x (m)')
    axes[1].set_ylabel('v (m/s)')
    axes[1].set_title('Phase Space (spiral inward)', fontweight='bold')
    axes[1].legend()

    # Energy
    KE = 0.5 * m * v**2
    PE = 0.5 * k * x**2
    E_total = KE + PE
    axes[2].plot(t, KE, 'dodgerblue', lw=1.5, label='KE')
    axes[2].plot(t, PE, 'orangered', lw=1.5, label='PE')
    axes[2].plot(t, E_total, 'green', lw=2, label='Total E')
    axes[2].set_xlabel('Time (s)')
    axes[2].set_ylabel('Energy (J)')
    axes[2].set_title('Energy Decay', fontweight='bold')
    axes[2].legend()

    fig.suptitle(f'b={b:.1f} N·s/m, k={k:.0f} N/m, m={m:.1f} kg, $\\gamma$={gamma:.2f}, $\\omega_0$={omega0:.2f}',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(interactive_damping,
    b=widgets.FloatSlider(value=0.5, min=0.0, max=15.0, step=0.1,
                          description='Damping b (N s/m):', style={'description_width': 'initial'}),
    k=widgets.FloatSlider(value=10.0, min=1.0, max=50.0, step=1.0,
                          description='k (N/m):', style={'description_width': 'initial'}),
    m=widgets.FloatSlider(value=1.0, min=0.1, max=5.0, step=0.1,
                          description='m (kg):', style={'description_width': 'initial'}),
    A=widgets.FloatSlider(value=1.0, min=0.2, max=3.0, step=0.1,
                          description='Amplitude (m):', style={'description_width': 'initial'})
);

---
## 10. Numerical Methods: Euler vs Analytical vs odeint

The **Euler method** is the simplest numerical ODE solver. For a second-order ODE rewritten as two first-order equations:

$$\dot{x} = v, \qquad \dot{v} = -\omega^2 x$$

The Euler update rule is:

$$x_{n+1} = x_n + v_n \cdot \Delta t$$
$$v_{n+1} = v_n + a_n \cdot \Delta t$$

This is like predicting tomorrow's weather using only today's rate of change. Simple but accumulates errors over time.

`scipy.integrate.odeint` uses adaptive step-size methods (LSODA) that are far more accurate.

---
## 11. Interactive Demo 5: Euler Method vs Analytical Solution

In [ ]:
def interactive_euler(dt, omega, n_periods):
    """Compare Euler method, odeint, and analytical solution."""
    A = 1.0
    T = 2 * np.pi / omega
    t_end = n_periods * T

    # --- Euler method ---
    t_euler = np.arange(0, t_end, dt)
    x_euler = np.zeros(len(t_euler))
    v_euler = np.zeros(len(t_euler))
    x_euler[0] = A
    v_euler[0] = 0

    for i in range(len(t_euler) - 1):
        a = -omega**2 * x_euler[i]
        x_euler[i + 1] = x_euler[i] + v_euler[i] * dt
        v_euler[i + 1] = v_euler[i] + a * dt

    # --- odeint ---
    def shm_ode(state, t):
        return [state[1], -omega**2 * state[0]]

    t_fine = np.linspace(0, t_end, 2000)
    sol = odeint(shm_ode, [A, 0], t_fine)
    x_odeint = sol[:, 0]

    # --- Analytical ---
    x_analytical = A * np.cos(omega * t_fine)
    x_analytical_euler = A * np.cos(omega * t_euler)

    fig, axes = plt.subplots(2, 2, figsize=(13, 8))

    # x(t) comparison
    axes[0, 0].plot(t_fine, x_analytical, 'b-', lw=2, label='Analytical', alpha=0.7)
    axes[0, 0].plot(t_euler, x_euler, 'r-', lw=1.5, label=f'Euler (dt={dt})')
    axes[0, 0].plot(t_fine, x_odeint, 'g--', lw=2, label='odeint', alpha=0.7)
    axes[0, 0].set_xlabel('Time (s)')
    axes[0, 0].set_ylabel('x (m)')
    axes[0, 0].set_title('Displacement Comparison', fontweight='bold')
    axes[0, 0].legend(fontsize=9)

    # Error
    error_euler = np.abs(x_euler - x_analytical_euler)
    axes[0, 1].plot(t_euler, error_euler, 'r-', lw=2, label='Euler error')
    axes[0, 1].set_xlabel('Time (s)')
    axes[0, 1].set_ylabel('|Error| (m)')
    axes[0, 1].set_title(f'Euler Error (max={error_euler.max():.4f} m)', fontweight='bold')
    axes[0, 1].legend()

    # Phase space
    axes[1, 0].plot(x_euler, v_euler, 'r-', lw=1.5, label='Euler', alpha=0.7)
    circle_theta = np.linspace(0, 2 * np.pi, 200)
    axes[1, 0].plot(A * np.cos(circle_theta), -A * omega * np.sin(circle_theta),
                    'b--', lw=2, label='Analytical', alpha=0.5)
    axes[1, 0].set_xlabel('x (m)')
    axes[1, 0].set_ylabel('v (m/s)')
    axes[1, 0].set_title('Phase Space', fontweight='bold')
    axes[1, 0].set_aspect('equal')
    axes[1, 0].legend()

    # Energy
    KE_euler = 0.5 * v_euler**2
    PE_euler = 0.5 * omega**2 * x_euler**2
    E_euler = KE_euler + PE_euler
    axes[1, 1].plot(t_euler, E_euler, 'r-', lw=2, label='Euler total E')
    axes[1, 1].axhline(y=0.5 * omega**2 * A**2, color='b', ls='--', lw=2,
                        label='True total E')
    axes[1, 1].set_xlabel('Time (s)')
    axes[1, 1].set_ylabel('Energy (arb. units)')
    axes[1, 1].set_title('Energy Conservation', fontweight='bold')
    axes[1, 1].legend()

    fig.suptitle(f'Euler Method: dt = {dt:.4f} s, $\\omega$ = {omega:.2f} rad/s, {n_periods} periods',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(interactive_euler,
    dt=widgets.FloatSlider(value=0.05, min=0.001, max=0.2, step=0.005,
                           description='dt (s):', style={'description_width': 'initial'},
                           readout_format='.3f'),
    omega=widgets.FloatSlider(value=2*np.pi, min=1.0, max=10.0, step=0.5,
                              description='$\\omega$ (rad/s):', style={'description_width': 'initial'}),
    n_periods=widgets.IntSlider(value=5, min=1, max=20, step=1,
                                description='Periods:', style={'description_width': 'initial'})
);

---
## 12. Worked Examples

### Example 1: Spring-Mass System

A 2.0 kg block is attached to a spring with $k = 200$ N/m. It is pulled 0.1 m from equilibrium and released from rest. Find (a) the angular frequency, (b) the period, (c) the maximum speed, (d) the maximum acceleration.

In [ ]:
m = 2.0      # kg
k = 200.0    # N/m
A = 0.1      # m

omega = np.sqrt(k / m)
T = 2 * np.pi / omega
v_max = A * omega
a_max = A * omega**2

print("=" * 50)
print("Worked Example 1: Spring-Mass System")
print("=" * 50)
print(f"(a) Angular frequency: omega = sqrt(k/m) = sqrt({k}/{m}) = {omega:.2f} rad/s")
print(f"(b) Period: T = 2*pi/omega = {T:.4f} s")
print(f"    Frequency: f = 1/T = {1/T:.2f} Hz")
print(f"(c) Max speed: v_max = A*omega = {A}*{omega:.2f} = {v_max:.2f} m/s")
print(f"(d) Max acceleration: a_max = A*omega^2 = {A}*{omega:.2f}^2 = {a_max:.2f} m/s^2")

### Example 2: Simple Pendulum Clock

A pendulum clock has a period of exactly 2.0 s (a "seconds pendulum"). Find the required length. What happens if the clock is taken to the Moon ($g_{\text{Moon}} = 1.62$ m/s$^2$)?

In [ ]:
T_target = 2.0  # s
g_earth = 9.81  # m/s^2
g_moon = 1.62   # m/s^2

# T = 2*pi*sqrt(L/g) => L = g*(T/(2*pi))^2
L_earth = g_earth * (T_target / (2 * np.pi))**2
T_moon = 2 * np.pi * np.sqrt(L_earth / g_moon)

print("=" * 50)
print("Worked Example 2: Pendulum Clock")
print("=" * 50)
print(f"Required length on Earth: L = {L_earth:.4f} m = {L_earth*100:.2f} cm")
print(f"\nWith the same pendulum on the Moon:")
print(f"Period on Moon: T = 2*pi*sqrt({L_earth:.4f}/{g_moon}) = {T_moon:.2f} s")
print(f"The clock would run {T_moon/T_target:.2f}x slower on the Moon!")

### Example 3: Damped Oscillation Energy Loss

A damped oscillator has $m = 0.5$ kg, $k = 20$ N/m, $b = 0.4$ N$\cdot$s/m. Starting from $x_0 = 0.3$ m at rest, after how many oscillations does the energy drop to half its initial value?

---
## Problem Set

**Instructions:** Solve each problem analytically first, then verify your answer numerically in the code cell below it. Show your work with clear variable definitions and unit tracking.

- **L1 (Basic):** Straightforward single-concept problems
- **L2 (Intermediate):** Multi-step problems combining two concepts
- **L3 (Challenge):** Multi-concept integration and engineering applications

### L1 (Basic) — P1

A $0.50$ kg mass is attached to a spring with $k = 200$ N/m. It is displaced $0.08$ m from equilibrium and released from rest. Find (a) the angular frequency, (b) the period, (c) the maximum speed, and (d) the maximum acceleration.

<details><summary>Answer</summary>(a) $\omega = 20$ rad/s; (b) $T = 0.314$ s; (c) $v_\text{max} = 1.60$ m/s; (d) $a_\text{max} = 32.0$ m/s$^2$</details>

In [ ]:
# ✏️ [P1] Your solution here

### L1 (Basic) — P2

A simple pendulum has a period of $2.50$ s on Earth ($g = 9.81$ m/s$^2$). What is its length?

<details><summary>Answer</summary>$L = 1.553$ m</details>

In [ ]:
# ✏️ [P2] Your solution here

### L1 (Basic) — P3

A block on a spring oscillates with amplitude $A = 0.12$ m and period $T = 0.80$ s. At what displacement from equilibrium is the speed half of its maximum value?

<details><summary>Answer</summary>$x = \pm 0.1039$ m</details>

In [ ]:
# ✏️ [P3] Your solution here

### L1 (Basic) — P4

A spring-mass system has $k = 50$ N/m, $m = 2.0$ kg, and amplitude $A = 0.20$ m. Find the total mechanical energy and the kinetic and potential energies when the displacement is $x = 0.10$ m.

<details><summary>Answer</summary>$E = 1.00$ J; $KE = 0.75$ J; $PE = 0.25$ J</details>

In [ ]:
# ✏️ [P4] Your solution here

### L2 (Intermediate) — P5

A $0.30$ kg mass on a spring ($k = 120$ N/m) is released from rest at $x = 0.15$ m. The system has a damping coefficient $b = 0.90$ N$\cdot$s/m. Find (a) the damping ratio $\gamma/\omega_0$, (b) the damped angular frequency $\omega_d$, and (c) the time for the amplitude to decay to $1/e$ of its initial value.

<details><summary>Answer</summary>(a) $\gamma/\omega_0 = 0.075$ (underdamped); (b) $\omega_d = 19.94$ rad/s; (c) $t = 0.667$ s</details>

In [ ]:
# ✏️ [P5] Your solution here

### L2 (Intermediate) — P6

A $1.2$ kg block oscillates on a spring ($k = 48$ N/m) with damping $b = 2.4$ N$\cdot$s/m. Starting from $x_0 = 0.25$ m at rest, find (a) the displacement after exactly $5$ complete oscillation cycles and (b) the fraction of mechanical energy remaining at that time.

<details><summary>Answer</summary>$\gamma = b/2m = 1.0$ s$^{-1}$, $\omega_d = 6.2450$ rad/s, $T_d = 1.00611$ s, so $5T_d = 5.0306$ s. (a) $x(5T_d) = 0.25e^{-5.0306} = 1.634$ mm; (b) $E/E_0 = e^{-2\gamma t} = 4.27\times10^{-5} = 0.00427\%$. (At a whole number of damped periods the exact solution reduces to $x_0e^{-\gamma t}$.) **[CORRECTED]** previously 0.0046 m and 0.034%, which correspond to $\gamma t \approx 4.0$ rather than 5.03.</details>

In [ ]:
# ✏️ [P6] Your solution here

### L2 (Intermediate) — P7

Two springs with constants $k_1 = 300$ N/m and $k_2 = 500$ N/m are connected in parallel to a $4.0$ kg block on a frictionless surface. The block is displaced $0.06$ m and released. Find the period of oscillation, the maximum speed, and the maximum kinetic energy.

<details><summary>Answer</summary>$T = 0.444$ s; $v_\text{max} = 0.849$ m/s; $KE_\text{max} = 1.44$ J</details>

In [ ]:
# ✏️ [P7] Your solution here

### L2 (Intermediate) — P8

A physical (compound) pendulum consists of a uniform thin rod of mass $m = 1.5$ kg and length $L = 0.80$ m, pivoted at a point $0.20$ m from one end. Find (a) the moment of inertia about the pivot, (b) the distance from pivot to center of mass, and (c) the period of small oscillations.

<details><summary>Answer</summary>(a) $I = I_\text{cm} + md^2 = \dfrac{mL^2}{12} + md^2 = 0.0800 + 1.5(0.20)^2 = 0.1400$ kg$\cdot$m$^2$; (b) $d = L/2 - 0.20 = 0.20$ m; (c) $T = 2\pi\sqrt{I/(mgd)} = 2\pi\sqrt{0.140/2.943} = 1.370$ s. **[CORRECTED]** previously $I = 0.0900$ (the parallel-axis term was all but dropped) and $T = 1.10$ s.</details>

In [ ]:
# ✏️ [P8] Your solution here

### L3 (Challenge) — P9

A vibration isolation mount for a sensitive instrument consists of a spring ($k = 8000$ N/m) and a viscous damper ($b = 120$ N$\cdot$s/m) supporting a $20$ kg platform. The platform is disturbed and oscillates freely. (a) Is the system underdamped, critically damped, or overdamped? (b) Find the damped frequency. (c) How long does it take for the vibration amplitude to drop below $0.1$ mm if the initial displacement is $5.0$ mm? (d) What value of $b$ would make the system critically damped?

<details><summary>Answer</summary>(a) Underdamped ($\gamma/\omega_0 = 0.15$); (b) $f_d = 3.14$ Hz; (c) $t = 1.30$ s; (d) $b_\text{crit} = 800$ N$\cdot$s/m</details>

In [ ]:
# ✏️ [P9] Your solution here

### L3 (Challenge) — P10

A robot arm joint is modeled as a torsional spring-damper system with torsional stiffness $\kappa = 50$ N$\cdot$m/rad, damping coefficient $c = 2.0$ N$\cdot$m$\cdot$s/rad, and moment of inertia $I = 0.40$ kg$\cdot$m$^2$ about the joint axis. The joint is displaced by $10^\circ$ and released. The governing equation is $I\ddot{\theta} + c\dot{\theta} + \kappa\theta = 0$. (a) Find the natural frequency and damping ratio. (b) Find the angular position $\theta(t)$ at $t = 0.5$ s. (c) Determine the settling time (time for the amplitude envelope to reach $2\%$ of initial).

<details><summary>Answer</summary>(a) $\omega_0 = 11.18$ rad/s, $\zeta = 0.224$; (b) $\theta(0.5) \approx -0.0434$ rad ($-2.49^\circ$); (c) $t_s \approx 1.56$ s</details>

In [ ]:
# ✏️ [P10] Your solution here

---
## 13. Bridge to Next Week

This week we studied **free oscillations** -- systems that oscillate naturally without external forcing. We saw how:

- Ideal SHM produces sinusoidal motion with constant amplitude
- Damping causes the amplitude to decay exponentially
- The phase space shrinks from a closed ellipse to an inward spiral

**Next week (Week 12)** we will explore what happens when you **drive** an oscillator with a periodic external force:

$$m\ddot{x} + b\dot{x} + kx = F_0 \cos(\omega_d t)$$

This leads to the fascinating phenomenon of **resonance** -- when the driving frequency matches the natural frequency, the amplitude grows dramatically. You will learn about:

- Amplitude-frequency response curves
- The quality factor $Q$ and its role in sharpening resonance peaks
- Phase lag between driving force and response
- Engineering applications: vibration dampers, seismic isolation, tuned circuits

**Prepare by thinking about**: Where have you encountered resonance in real life? (Hint: pushing a swing, tuning a radio, wine glass shattering...)